# 01_Load_PMC_Inventory

Tareas:
1. Consulta PMC mediante NCBI ESearch.
2. Obtiene los PMCIDs del dominio **Rare Genetic Diseases**.
3. Enriquece los artículos con los metadatos bibliográficos realmente utilizados por la solución.
4. Identifica las versiones disponibles en `pmc-oa-opendata`.
5. Recupera las URLs de PDF y XML/JATS.
6. Persiste el inventario en Delta / Unity Catalog mediante un `MERGE` idempotente.
7. Registra la ejecución en `pipeline_runs`.

In [0]:
# ============================================================
# Librerías
# ============================================================
from __future__ import annotations

import json
import time
import uuid
from datetime import datetime, timezone
from typing import Any

import boto3
import requests
from botocore import UNSIGNED
from botocore.config import Config
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T


In [0]:
# ============================================================
# Configuración
# ============================================================

CORPUS_DOMAIN = "Rare Genetic Diseases"

SEARCH_TERM = (
    '("rare disease"[Title/Abstract] OR "rare diseases"[Title/Abstract] '
    'OR "rare genetic disease"[Title/Abstract] '
    'OR "rare genetic diseases"[Title/Abstract]) '
    'AND '
    '(genetic*[Title/Abstract] OR inherited[Title/Abstract] '
    'OR hereditary[Title/Abstract] OR genomic*[Title/Abstract])'
)

# Durante la auditoría E2E trabajaremos inicialmente con un corpus pequeño.
# Para la ejecución final del TFM se aumentará este valor y se congelará el corpus.
MAX_RECORDS = 30

# IMPORTANTE: True elimina TODO el esquema Databricks del pipeline y lo recrea.
# Úselo únicamente para esta reconstrucción controlada.
# Después de la primera ejecución limpia, cambiar a False.
RESET_PIPELINE_SCHEMA = False

NCBI_TOOL = "pmc_tfm_pipeline"
NCBI_EMAIL = "luisjose1327@gmail.com"

PMC_BUCKET = "pmc-oa-opendata"
PMC_HTTPS_BASE = "https://pmc-oa-opendata.s3.amazonaws.com"

CATALOG_NAME = "workspace"
SCHEMA_NAME = "tfm_pmc"

TARGET_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_inventory"
PIPELINE_RUNS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pipeline_runs"

PIPELINE_NAME = "01_Load_PMC_Inventory_v3"
RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

REQUEST_TIMEOUT_SECONDS = 60
MAX_REQUEST_RETRIES = 4
REQUEST_DELAY_SECONDS = 0.15
ESUMMARY_BATCH_SIZE = 100

query = (
    f"({SEARCH_TERM}) "
    "AND open_access[filter] "
    "AND has_pdf[filter]"
)

print(f"Run ID: {RUN_ID}")
print(f"Corpus domain: {CORPUS_DOMAIN}")
print(f"Maximum records: {MAX_RECORDS}")
print(f"Reset pipeline schema: {RESET_PIPELINE_SCHEMA}")
print(f"Target table: {TARGET_TABLE}")
print(f"PMC query: {query}")


Run ID: 0c111263-8523-4ed4-9de1-cd66be46c1db
Corpus domain: Rare Genetic Diseases
Maximum records: 20
Reset pipeline schema: True
Target table: workspace.tfm_pmc.pmc_inventory
PMC query: (("rare disease"[Title/Abstract] OR "rare diseases"[Title/Abstract] OR "rare genetic disease"[Title/Abstract] OR "rare genetic diseases"[Title/Abstract]) AND (genetic*[Title/Abstract] OR inherited[Title/Abstract] OR hereditary[Title/Abstract] OR genomic*[Title/Abstract])) AND open_access[filter] AND has_pdf[filter]


In [0]:
# ============================================================
# Reinicio controlado del entorno Databricks
# ============================================================
# Esta operación NO modifica Supabase.
# Se utiliza ahora porque estamos cambiando el contrato de datos entre etapas.

if RESET_PIPELINE_SCHEMA:
    print(f"Dropping schema {CATALOG_NAME}.{SCHEMA_NAME} CASCADE ...")
    spark.sql(
        f"DROP SCHEMA IF EXISTS {CATALOG_NAME}.{SCHEMA_NAME} CASCADE"
    )

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}"
)

print("Databricks schema ready.")


Dropping schema workspace.tfm_pmc CASCADE ...
Databricks schema ready.


In [0]:
# ============================================================
# Tablas persistentes
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        pmcid STRING NOT NULL,
        article_version STRING NOT NULL,
        version INT,
        is_latest_version BOOLEAN,

        pmid STRING,
        doi STRING,

        title STRING,
        citation STRING,
        journal STRING,
        publication_date STRING,
        publication_year INT,
        author_names ARRAY<STRING>,

        is_retracted BOOLEAN,
        license_code STRING,

        pdf_url STRING,
        xml_url STRING,

        corpus_domain STRING,

        download_status STRING,
        extraction_status STRING,
        cleaning_status STRING,
        chunking_status STRING,
        publication_status STRING,

        inventory_run_id STRING,
        ingested_at TIMESTAMP,
        updated_at TIMESTAMP,
        error_message STRING
    )
    USING DELTA
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {PIPELINE_RUNS_TABLE} (
        run_id STRING NOT NULL,
        pipeline_name STRING NOT NULL,
        run_status STRING NOT NULL,
        started_at TIMESTAMP NOT NULL,
        completed_at TIMESTAMP,
        records_requested BIGINT,
        records_found BIGINT,
        records_processed BIGINT,
        records_inserted BIGINT,
        records_updated BIGINT,
        records_failed BIGINT,
        execution_metadata STRING,
        error_message STRING
    )
    USING DELTA
    """
)

print("Persistent tables ready.")


Persistent tables ready.


In [0]:
# ============================================================
# Cliente HTTP con reintentos
# ============================================================

def request_json(
    url: str,
    params: dict[str, Any] | None = None,
    max_retries: int = MAX_REQUEST_RETRIES,
) -> dict[str, Any]:
    headers = {
        "User-Agent": f"{NCBI_TOOL}/1.0 ({NCBI_EMAIL})",
        "Accept": "application/json",
    }

    last_error: Exception | None = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", "2"))
                time.sleep(retry_after)
                continue

            response.raise_for_status()
            return response.json()

        except (requests.RequestException, ValueError) as error:
            last_error = error

            if attempt == max_retries:
                break

            wait_seconds = min(2 ** attempt, 15)
            print(
                f"HTTP attempt {attempt} failed. "
                f"Retrying in {wait_seconds} seconds."
            )
            time.sleep(wait_seconds)

    raise RuntimeError(
        f"Request failed after {max_retries} attempts: {url}"
    ) from last_error


In [0]:
# ============================================================
# Consulta PMC mediante ESearch
# ============================================================

def search_pmc(
    query: str,
    max_records: int,
    retstart: int = 0,
) -> dict[str, Any]:
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

    params = {
        "db": "pmc",
        "term": query,
        "retmode": "json",
        "retstart": retstart,
        "retmax": max_records,
        "sort": "date",
        "tool": NCBI_TOOL,
        "email": NCBI_EMAIL,
    }

    return request_json(url=url, params=params)


search_response = search_pmc(
    query=query,
    max_records=MAX_RECORDS,
)

search_result = search_response["esearchresult"]
total_available = int(search_result["count"])
numeric_ids = search_result["idlist"]
pmcids = [f"PMC{numeric_id}" for numeric_id in numeric_ids]

if not pmcids:
    raise RuntimeError(
        "PMC did not return records for the configured query."
    )

print("Total matching articles:", total_available)
print("Retrieved in this execution:", len(pmcids))
print("PMCIDs:", pmcids)


Total matching articles: 4521
Retrieved in this execution: 20
PMCIDs: ['PMC13570588', 'PMC13570527', 'PMC13564229', 'PMC13561766', 'PMC13564258', 'PMC13569294', 'PMC13559883', 'PMC13560980', 'PMC13554935', 'PMC13552808', 'PMC13552502', 'PMC13547469', 'PMC13547081', 'PMC13545448', 'PMC13540131', 'PMC13544149', 'PMC13543812', 'PMC13539412', 'PMC13538279', 'PMC13538227']


In [0]:
# ============================================================
# Enriquecimiento bibliográfico mediante ESummary
# ============================================================

def chunked(values: list[str], size: int):
    for start in range(0, len(values), size):
        yield values[start:start + size]


def extract_year(date_value: str | None) -> int | None:
    if not date_value:
        return None

    for token in str(date_value).replace("-", " ").split():
        if len(token) == 4 and token.isdigit():
            year = int(token)
            if 1800 <= year <= 2100:
                return year

    return None


def fetch_pmc_esummary(
    pmcids: list[str],
) -> dict[str, dict[str, Any]]:
    url = (
        "https://eutils.ncbi.nlm.nih.gov/"
        "entrez/eutils/esummary.fcgi"
    )

    metadata_by_pmcid: dict[str, dict[str, Any]] = {}

    for batch in chunked(pmcids, ESUMMARY_BATCH_SIZE):
        numeric_ids = [
            pmcid.removeprefix("PMC")
            for pmcid in batch
        ]

        response = request_json(
            url=url,
            params={
                "db": "pmc",
                "id": ",".join(numeric_ids),
                "retmode": "json",
                "tool": NCBI_TOOL,
                "email": NCBI_EMAIL,
            },
        )

        result = response.get("result", {})

        for numeric_id in numeric_ids:
            summary = result.get(numeric_id, {})
            pmcid = f"PMC{numeric_id}"

            publication_date = (
                summary.get("pubdate")
                or summary.get("epubdate")
                or None
            )

            author_names = [
                str(author["name"])
                for author in (summary.get("authors") or [])
                if author.get("name")
            ]

            metadata_by_pmcid[pmcid] = {
                "journal": (
                    summary.get("fulljournalname")
                    or summary.get("source")
                ),
                "publication_date": publication_date,
                "publication_year": extract_year(publication_date),
                "author_names": author_names,
            }

        time.sleep(REQUEST_DELAY_SECONDS)

    return metadata_by_pmcid


esummary_by_pmcid = fetch_pmc_esummary(pmcids)

print(
    "PMCIDs enriched with ESummary:",
    len(esummary_by_pmcid),
)


PMCIDs enriched with ESummary: 20


In [0]:
# ============================================================
# Cliente anónimo para el bucket público de PMC
# ============================================================

s3_client = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED),
)


def get_article_versions(pmcid: str) -> list[str]:
    response = s3_client.list_objects_v2(
        Bucket=PMC_BUCKET,
        Prefix=f"{pmcid}.",
        Delimiter="/",
    )

    return [
        item["Prefix"].rstrip("/")
        for item in response.get("CommonPrefixes", [])
    ]


def get_version_number(article_version: str) -> int:
    return int(article_version.rsplit(".", 1)[1])


def get_latest_version(versions: list[str]) -> str | None:
    if not versions:
        return None

    return max(versions, key=get_version_number)


In [0]:
# ============================================================
# Metadatos oficiales de PMC OA
# ============================================================

def get_metadata_url(article_version: str) -> str:
    return (
        f"{PMC_HTTPS_BASE}/metadata/"
        f"{article_version}.json"
    )


def fetch_article_metadata(
    article_version: str,
) -> dict[str, Any]:
    return request_json(get_metadata_url(article_version))


def s3_url_to_https(url: str | None) -> str | None:
    if not url:
        return None

    s3_prefix = f"s3://{PMC_BUCKET}/"

    if url.startswith(s3_prefix):
        object_path = url[len(s3_prefix):]
        return f"{PMC_HTTPS_BASE}/{object_path}"

    return url


def build_inventory_record(
    pmcid: str,
    article_version: str,
    is_latest_version: bool,
) -> dict[str, Any]:
    metadata = fetch_article_metadata(article_version)
    bibliographic = esummary_by_pmcid.get(pmcid, {})
    now = datetime.now(timezone.utc).isoformat()

    return {
        "pmcid": metadata.get("pmcid", pmcid),
        "article_version": article_version,
        "version": int(
            metadata.get(
                "version",
                get_version_number(article_version),
            )
        ),
        "is_latest_version": is_latest_version,
        "pmid": (
            str(metadata["pmid"])
            if metadata.get("pmid") is not None
            else None
        ),
        "doi": metadata.get("doi"),
        "title": metadata.get("title"),
        "citation": metadata.get("citation"),
        "journal": bibliographic.get("journal"),
        "publication_date": bibliographic.get("publication_date"),
        "publication_year": bibliographic.get("publication_year"),
        "author_names": bibliographic.get("author_names") or [],
        "is_retracted": metadata.get("is_retracted"),
        "license_code": metadata.get("license_code"),
        "pdf_url": s3_url_to_https(metadata.get("pdf_url")),
        "xml_url": s3_url_to_https(metadata.get("xml_url")),
        "corpus_domain": CORPUS_DOMAIN,
        "inventory_run_id": RUN_ID,
        "ingested_at": now,
        "updated_at": now,
    }


In [0]:
# ============================================================
# Construcción del inventario
# ============================================================

inventory_records: list[dict[str, Any]] = []
failed_records: list[dict[str, str]] = []

for index, pmcid in enumerate(pmcids, start=1):
    try:
        versions = get_article_versions(pmcid)

        if not versions:
            failed_records.append({
                "pmcid": pmcid,
                "stage": "version_lookup",
                "error": "No article versions found",
            })
            print(
                f"[{index}/{len(pmcids)}] "
                f"{pmcid}: no versions found"
            )
            continue

        latest_version = get_latest_version(versions)

        for article_version in versions:
            inventory_records.append(
                build_inventory_record(
                    pmcid=pmcid,
                    article_version=article_version,
                    is_latest_version=(
                        article_version == latest_version
                    ),
                )
            )

        print(
            f"[{index}/{len(pmcids)}] "
            f"{pmcid}: {len(versions)} version(s)"
        )
        time.sleep(REQUEST_DELAY_SECONDS)

    except Exception as error:
        failed_records.append({
            "pmcid": pmcid,
            "stage": "metadata_ingestion",
            "error": str(error),
        })
        print(
            f"[{index}/{len(pmcids)}] "
            f"{pmcid}: FAILED - {error}"
        )

if not inventory_records:
    raise RuntimeError(
        "No inventory records were successfully retrieved."
    )

print("Inventory records:", len(inventory_records))
print("Failed PMCIDs:", len(failed_records))


[1/20] PMC13570588: 1 version(s)
[2/20] PMC13570527: 1 version(s)
[3/20] PMC13564229: 1 version(s)
[4/20] PMC13561766: 1 version(s)
[5/20] PMC13564258: 1 version(s)
[6/20] PMC13569294: 1 version(s)
[7/20] PMC13559883: 1 version(s)
[8/20] PMC13560980: 1 version(s)
[9/20] PMC13554935: 1 version(s)
[10/20] PMC13552808: 1 version(s)
[11/20] PMC13552502: 1 version(s)
[12/20] PMC13547469: 1 version(s)
[13/20] PMC13547081: 1 version(s)
[14/20] PMC13545448: 1 version(s)
[15/20] PMC13540131: 2 version(s)
[16/20] PMC13544149: 1 version(s)
[17/20] PMC13543812: 1 version(s)
[18/20] PMC13539412: 1 version(s)
[19/20] PMC13538279: 1 version(s)
[20/20] PMC13538227: 1 version(s)
Inventory records: 21
Failed PMCIDs: 0


In [0]:
# ============================================================
# DataFrame normalizado
# ============================================================

inventory_schema = T.StructType([
    T.StructField("pmcid", T.StringType(), False),
    T.StructField("article_version", T.StringType(), False),
    T.StructField("version", T.IntegerType(), True),
    T.StructField("is_latest_version", T.BooleanType(), True),
    T.StructField("pmid", T.StringType(), True),
    T.StructField("doi", T.StringType(), True),
    T.StructField("title", T.StringType(), True),
    T.StructField("citation", T.StringType(), True),
    T.StructField("journal", T.StringType(), True),
    T.StructField("publication_date", T.StringType(), True),
    T.StructField("publication_year", T.IntegerType(), True),
    T.StructField(
        "author_names",
        T.ArrayType(T.StringType()),
        True,
    ),
    T.StructField("is_retracted", T.BooleanType(), True),
    T.StructField("license_code", T.StringType(), True),
    T.StructField("pdf_url", T.StringType(), True),
    T.StructField("xml_url", T.StringType(), True),
    T.StructField("corpus_domain", T.StringType(), True),
    T.StructField("inventory_run_id", T.StringType(), False),
    T.StructField("ingested_at", T.StringType(), False),
    T.StructField("updated_at", T.StringType(), False),
])

inventory_df = (
    spark.createDataFrame(
        inventory_records,
        schema=inventory_schema,
    )
    .withColumn(
        "ingested_at",
        F.to_timestamp("ingested_at"),
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at"),
    )
    .withColumn("download_status", F.lit("pending"))
    .withColumn("extraction_status", F.lit("pending"))
    .withColumn("cleaning_status", F.lit("pending"))
    .withColumn("chunking_status", F.lit("pending"))
    .withColumn("publication_status", F.lit("pending"))
    .withColumn("error_message", F.lit(None).cast("string"))
    .select(
        "pmcid",
        "article_version",
        "version",
        "is_latest_version",
        "pmid",
        "doi",
        "title",
        "citation",
        "journal",
        "publication_date",
        "publication_year",
        "author_names",
        "is_retracted",
        "license_code",
        "pdf_url",
        "xml_url",
        "corpus_domain",
        "download_status",
        "extraction_status",
        "cleaning_status",
        "chunking_status",
        "publication_status",
        "inventory_run_id",
        "ingested_at",
        "updated_at",
        "error_message",
    )
)

display(inventory_df)


pmcid,article_version,version,is_latest_version,pmid,doi,title,citation,journal,publication_date,publication_year,author_names,is_retracted,license_code,pdf_url,xml_url,corpus_domain,download_status,extraction_status,cleaning_status,chunking_status,publication_status,inventory_run_id,ingested_at,updated_at,error_message
PMC13570588,PMC13570588.1,1,true,42731022,10.1093/ecco-jcc/jjag124,A therapeutic atlas of monogenic inflammatory bowel disease,J Crohns Colitis. 2026 Sep 12;20(9):jjag124. doi: 10.1093/ecco-jcc/jjag124,Journal of Crohn's & colitis,2026 Sep 3,2026,"List(Yeh PJ, Charlesworth JEG, Taylor H, Ashton JJ, Nash K, Lam KH, de Ridder L, Vuijk SA, Ye Z, Huang Y, Bildstein T, Haller W, Jones KDJ, Shouval DS, Weiss B, Lau YL, Bui-Thi-Thuy Q, Muise AM, Richards D, Travis S, Turner D, Uhlig HH)",false,CC BY,https://pmc-oa-opendata.s3.amazonaws.com/PMC13570588.1/PMC13570588.1.pdf?md5=a929ddba6ca870f6d7c0ef35f55c2733,https://pmc-oa-opendata.s3.amazonaws.com/PMC13570588.1/PMC13570588.1.xml?md5=0c16aa5e2049b51577e70f8a802d8803,Rare Genetic Diseases,pending,pending,pending,pending,pending,0c111263-8523-4ed4-9de1-cd66be46c1db,2026-09-13T20:11:16.116Z,2026-09-13T20:11:16.116Z,null
PMC13570527,PMC13570527.1,1,true,null,10.1186/s13023-026-04323-4,Diagnosis of rare diseases based on facial phenotype: a quantitative assessment using 2D and 3D photography in Stickler syndrome,Orphanet J Rare Dis. 2026 Jul 27;21:317. doi: 10.1186/s13023-026-04323-4,Orphanet Journal of Rare Diseases,2026,2026,"List(Rohée-Traoré A, Taverne M, Hennocq Q, Bongibault T, Daruich A, Bremond-Gignac D, Kün-Darbois JD, Rothschild PR, Khonsari RH)",false,CC BY,https://pmc-oa-opendata.s3.amazonaws.com/PMC13570527.1/PMC13570527.1.pdf?md5=02e7ccdc38503160ae02757cc37b1ba7,https://pmc-oa-opendata.s3.amazonaws.com/PMC13570527.1/PMC13570527.1.xml?md5=bb9cc36eee0d19a10ec43c412c77955b,Rare Genetic Diseases,pending,pending,pending,pending,pending,0c111263-8523-4ed4-9de1-cd66be46c1db,2026-09-13T20:11:16.451Z,2026-09-13T20:11:16.451Z,null
PMC13564229,PMC13564229.1,1,true,42729987,10.1093/haschl/qxag238,Financing and health system capacity for precision medicine in Asia: a six country landscape analysis,Health Aff Sch. 2026 Sep 1;4(9):qxag238. doi: 10.1093/haschl/qxag238,Health affairs scholar,2026 Sep,2026,"List(Peh ALT, Teerawattananon Y, Yuen JXY, Charafi N, Tahapary DL, Leng J, Morton A, Li S, Kim WJ, Zilfalil BA, Hasbullah HH, Satproedprai N, Li ST, Ngeow JYY, Juang YR, Zhang Y, Hong M, Chen W)",false,CC BY,https://pmc-oa-opendata.s3.amazonaws.com/PMC13564229.1/PMC13564229.1.pdf?md5=727d7c671cf6262d3483b70beafb5abd,https://pmc-oa-opendata.s3.amazonaws.com/PMC13564229.1/PMC13564229.1.xml?md5=157a0453894c25a55306c15edd12c2d6,Rare Genetic Diseases,pending,pending,pending,pending,pending,0c111263-8523-4ed4-9de1-cd66be46c1db,2026-09-13T20:11:16.691Z,2026-09-13T20:11:16.691Z,null
PMC13561766,PMC13561766.1,1,true,42729137,10.3389/fimmu.2026.1824739,A case report of ADMIO type 1 caused by a de novo STAT3 gain-of-function mutation,Front Immunol. 2026 Aug 28;17:1824739. doi: 10.3389/fimmu.2026.1824739,Frontiers in immunology,2026,2026,"List(Huang L, Yue Y, Zhu Y, Hao H, Guan F)",false,CC BY,https://pmc-oa-opendata.s3.amazonaws.com/PMC13561766.1/PMC13561766.1.pdf?md5=73cca79857a4b6faeb11a12644e0b5bb,https://pmc-oa-opendata.s3.amazonaws.com/PMC13561766.1/PMC13561766.1.xml?md5=82fdcbb89daff579f9605fc3d60d858e,Rare Genetic Diseases,pending,pending,pending,pending,pending,0c111263-8523-4ed4-9de1-cd66be46c1db,2026-09-13T20:11:16.932Z,2026-09-13T20:11:16.932Z,null
PMC13564258,PMC13564258.1,1,true,42725902,10.70962/jhi.20260153,"In Memoriam: Daniel L. Kastner, MD (1951–2026)",J Hum Immun. 2026 Sep 11;2(6):e20260153. doi: 10.70962/jhi.20260153,Journal of human immunity,2026 Nov 2,2026,"List(Gupta S, Goldbach-Mansky R, Aksentijevich I)",false,CC BY,https://pmc-oa-opendata.s3.amazonaws.com/PMC13564258.1/PMC13564258.1.pdf?md5=b31bff55ad7e50a004007b71b5cfb5a6,https://pmc-oa-opendata.s3.amazonaws.

In [0]:
# ============================================================
# Validaciones de calidad
# ============================================================

quality_metrics_df = (
    inventory_df
    .agg(
        F.count("*").alias("total_article_versions"),
        F.countDistinct(
            F.struct("pmcid", "article_version")
        ).alias("distinct_article_versions"),
        F.countDistinct("pmcid").alias("unique_pmcids"),
        F.sum(
            F.when(F.col("pdf_url").isNotNull(), 1).otherwise(0)
        ).alias("records_with_pdf"),
        F.sum(
            F.when(F.col("xml_url").isNotNull(), 1).otherwise(0)
        ).alias("records_with_xml"),
        F.sum(
            F.when(F.col("publication_year").isNotNull(), 1).otherwise(0)
        ).alias("records_with_publication_year"),
        F.sum(
            F.when(F.size(F.col("author_names")) > 0, 1).otherwise(0)
        ).alias("records_with_authors"),
        F.sum(
            F.when(F.col("is_retracted") == True, 1).otherwise(0)
        ).alias("retracted_records"),
        F.sum(
            F.when(
                F.col("pmcid").isNull()
                | F.col("article_version").isNull(),
                1,
            ).otherwise(0)
        ).alias("null_business_keys"),
    )
)

quality = quality_metrics_df.first()

if quality["total_article_versions"] == 0:
    raise RuntimeError("The inventory DataFrame is empty.")

if quality["null_business_keys"] > 0:
    raise RuntimeError(
        "The inventory contains null business keys."
    )

if (
    quality["total_article_versions"]
    != quality["distinct_article_versions"]
):
    raise RuntimeError(
        "The inventory contains duplicate article versions."
    )

display(quality_metrics_df)


total_article_versions,distinct_article_versions,unique_pmcids,records_with_pdf,records_with_xml,records_with_publication_year,records_with_authors,retracted_records,null_business_keys
21,21,20,21,21,21,21,0,0


In [0]:
# ============================================================
# Métricas previas al MERGE
# ============================================================

target_keys_df = (
    spark.table(TARGET_TABLE)
    .select("pmcid", "article_version")
)

change_metrics = (
    inventory_df.alias("source")
    .join(
        target_keys_df.alias("target"),
        on=[
            F.col("source.pmcid") == F.col("target.pmcid"),
            F.col("source.article_version")
            == F.col("target.article_version"),
        ],
        how="left",
    )
    .agg(
        F.sum(
            F.when(F.col("target.pmcid").isNull(), 1).otherwise(0)
        ).alias("records_to_insert"),
        F.sum(
            F.when(F.col("target.pmcid").isNotNull(), 1).otherwise(0)
        ).alias("records_to_update"),
    )
    .first()
)

records_to_insert = int(
    change_metrics["records_to_insert"] or 0
)
records_to_update = int(
    change_metrics["records_to_update"] or 0
)

print(f"Records to insert: {records_to_insert}")
print(f"Records to update: {records_to_update}")


Records to insert: 21
Records to update: 0


In [0]:
# ============================================================
# MERGE idempotente
# ============================================================
# En registros existentes:
# - se actualizan únicamente metadatos del inventario;
# - se preservan ingested_at, estados posteriores y error_message.

inventory_delta = DeltaTable.forName(
    spark,
    TARGET_TABLE,
)

(
    inventory_delta.alias("target")
    .merge(
        inventory_df.alias("source"),
        """
        target.pmcid = source.pmcid
        AND target.article_version = source.article_version
        """,
    )
    .whenMatchedUpdate(
        set={
            "version": "source.version",
            "is_latest_version": "source.is_latest_version",
            "pmid": "source.pmid",
            "doi": "source.doi",
            "title": "source.title",
            "citation": "source.citation",
            "journal": "source.journal",
            "publication_date": "source.publication_date",
            "publication_year": "source.publication_year",
            "author_names": "source.author_names",
            "is_retracted": "source.is_retracted",
            "license_code": "source.license_code",
            "pdf_url": "source.pdf_url",
            "xml_url": "source.xml_url",
            "corpus_domain": "source.corpus_domain",
            "inventory_run_id": "source.inventory_run_id",
            "updated_at": "source.updated_at",
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Inventory MERGE completed successfully.")


Inventory MERGE completed successfully.


In [0]:
# ============================================================
# Registro de ejecución
# ============================================================

RUN_COMPLETED_AT = datetime.now(timezone.utc)

run_schema = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("pipeline_name", T.StringType(), False),
    T.StructField("run_status", T.StringType(), False),
    T.StructField("started_at", T.TimestampType(), False),
    T.StructField("completed_at", T.TimestampType(), True),
    T.StructField("records_requested", T.LongType(), True),
    T.StructField("records_found", T.LongType(), True),
    T.StructField("records_processed", T.LongType(), True),
    T.StructField("records_inserted", T.LongType(), True),
    T.StructField("records_updated", T.LongType(), True),
    T.StructField("records_failed", T.LongType(), True),
    T.StructField("execution_metadata", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
])

run_df = spark.createDataFrame(
    [(
        RUN_ID,
        PIPELINE_NAME,
        "completed",
        RUN_STARTED_AT,
        RUN_COMPLETED_AT,
        int(MAX_RECORDS),
        int(len(pmcids)),
        int(len(inventory_records)),
        records_to_insert,
        records_to_update,
        int(len(failed_records)),
        json.dumps({
            "corpus_domain": CORPUS_DOMAIN,
            "search_term": SEARCH_TERM,
            "search_query": query,
            "total_available": total_available,
            "metadata_source": "NCBI ESummary + PMC OA metadata",
            "reset_pipeline_schema": RESET_PIPELINE_SCHEMA,
        }),
        None,
    )],
    schema=run_schema,
)

run_df.write.mode("append").saveAsTable(
    PIPELINE_RUNS_TABLE
)

print(f"Pipeline run registered: {RUN_ID}")


Pipeline run registered: 0c111263-8523-4ed4-9de1-cd66be46c1db


In [0]:
# ============================================================
# Resultado final -- DELETE LATER
# ============================================================

summary_df = (
    spark.table(TARGET_TABLE)
    .agg(
        F.count("*").alias("total_article_versions"),
        F.countDistinct("pmcid").alias("unique_pmcids"),
        F.sum(
            F.when(F.col("pdf_url").isNotNull(), 1).otherwise(0)
        ).alias("records_with_pdf"),
        F.sum(
            F.when(F.col("xml_url").isNotNull(), 1).otherwise(0)
        ).alias("records_with_xml"),
        F.sum(
            F.when(F.col("publication_year").isNotNull(), 1).otherwise(0)
        ).alias("records_with_year"),
        F.sum(
            F.when(F.size(F.col("author_names")) > 0, 1).otherwise(0)
        ).alias("records_with_authors"),
        F.sum(
            F.when(F.col("download_status") == "pending", 1).otherwise(0)
        ).alias("pending_download"),
    )
)

display(summary_df)

display(
    spark.table(TARGET_TABLE)
    .select(
        "pmcid",
        "article_version",
        "title",
        "journal",
        "publication_year",
        "author_names",
        "pdf_url",
        "xml_url",
        "download_status",
    )
    .orderBy(
        F.col("publication_year").desc_nulls_last(),
        F.col("pmcid"),
    )
)


total_article_versions,unique_pmcids,records_with_pdf,records_with_xml,records_with_year,records_with_authors,pending_download
21,20,21,21,21,21,21


pmcid,article_version,title,journal,publication_year,author_names,pdf_url,xml_url,download_status
PMC13538227,PMC13538227.1,Hereditary connective tissue disorders in unselected patients with spontaneous cervical artery dissection: a targeted next generation sequencing approach and systematic review,Neurological sciences : official journal of the Italian Neurological Society and of the Italian Society of Clinical Neurophysiology,2026,"List(Corradi L, Ferraro C, Tesi F, Abrignani G, Castellini P, Latte L, Trapasso MC, Genovese A, Ritelli MG, Cinquina V, Giliani SC, Magoni M, Menozzi R, Pezzini A)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13538227.1/PMC13538227.1.pdf?md5=96dbbf17b74b615c55f31cca4749cbbe,https://pmc-oa-opendata.s3.amazonaws.com/PMC13538227.1/PMC13538227.1.xml?md5=07b94f2fe91d2f04a955e0bc9f0a5676,pending
PMC13538279,PMC13538279.1,Validation of a Cellular Imaging‐Based Method as a Potential Biomarker for SPG4 Hereditary Spastic Paraplegia,Annals of clinical and translational neurology,2026,"List(Fattorini G, Licursi V, Zanna GD, Dal Canto F, Barghigiani M, Setola N, Rossi S, Funcis A, Santorelli FM, Silvestri G, Casali C, Sardina F, Rinaldo C)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13538279.1/PMC13538279.1.pdf?md5=f14c06178b556380ad81d43bf68ec6bd,https://pmc-oa-opendata.s3.amazonaws.com/PMC13538279.1/PMC13538279.1.xml?md5=05ada6e7f32c1976a3acdd7d23ea5fbe,pending
PMC13539412,PMC13539412.1,Parathyroid carcinoma: epidemiology and genetics,"Endocrine oncology (Bristol, England)",2026,"List(Betea D, Petrossians P)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13539412.1/PMC13539412.1.pdf?md5=96e5e0f52e1c42e8be83353af60762b8,https://pmc-oa-opendata.s3.amazonaws.com/PMC13539412.1/PMC13539412.1.xml?md5=4af7f054b158ebeda2a40ad8eb693492,pending
PMC13540131,PMC13540131.2,Development of an electrochemiluminescence-based bridging assay to detect antibodies against a PTH inverse agonist in human plasma,Bioanalysis,2026,"List(Nduwumwami AJ, Wagner EJ, Wang AQ, Fang Y, Tao D, Xu X)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13540131.2/PMC13540131.2.pdf?md5=c1a90cd0c652661c702dc49e08a52106,https://pmc-oa-opendata.s3.amazonaws.com/PMC13540131.2/PMC13540131.2.xml?md5=0acdbdfaa29d5c4fe7759c6bf3663bcf,pending
PMC13540131,PMC13540131.1,Development of an electrochemiluminescence-based bridging assay to detect antibodies against a PTH inverse agonist in human plasma,Bioanalysis,2026,"List(Nduwumwami AJ, Wagner EJ, Wang AQ, Fang Y, Tao D, Xu X)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13540131.1/PMC13540131.1.pdf?md5=59906596f718bdd65017e0342dcd8b69,https://pmc-oa-opendata.s3.amazonaws.com/PMC13540131.1/PMC13540131.1.xml?md5=78c5d5a9add9a4269ff199c2d7210ed2,pending
PMC13543812,PMC13543812.1,"Equity in genome sequencing for rare disease diagnosis: a cross-sectional analysis of data from the UK 100,000 Genomes Project",EBioMedicine,2026,"List(Tallman S, Moutsianas L, Nguyen T, Cho Y, Mackintosh M, Kasperaviciute D, Brown MA, Ellingford JM, Kuchenbaecker K, Silver MJ)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13543812.1/PMC13543812.1.pdf?md5=6b4e0bb37b7a03d33a4e03f9f37bce1e,https://pmc-oa-opendata.s3.amazonaws.com/PMC13543812.1/PMC13543812.1.xml?md5=684de2fc9f0bad792f1cc7332aecdc77,pending
PMC13544149,PMC13544149.1,Identification of a Novel CDH2 Gene Variant in an ACOGS Patient with Concurrent Respiratory Tract Infection: A Case Report,"Pediatric health, medicine and therapeutics",2026,"List(Lu Y, Fang F, Zhou H, Shu S, Liu X)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13544149.1/PMC13544149.1.pdf?md5=a9897a9c2595c2ea928b9e549785b11e,https://pmc-oa-opendata.s3.amazonaws.com/PMC13544149.1/PMC13544149.1.xml?md5=4f0a22d17d6ae25ccbcc3aee7c27a2cf,pending
PMC13545448,PMC13545448.1,Artificial intelligence-assisted clinical exome sequencing: Insights and outcomes from 822 pediatric diagnoses,Genetics in medicine open,2026,"List(Pan Y, Danley P, Kramer T, Noruzinia M, Buser K, Yatsenko AN, Bellissimo D, Guo F)",https://pmc-oa-opendat